# VideoMAE

In [1]:
import torch
import model.videomae
import data.dataloader

/home/bendavison/PycharmProjects/Group-35-Applied-ML-Coursework/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data loading

In [2]:
cfg = {
        "num_frames": 8,
        "resolution": 224,
        "train_ratio": 0.70,
        "val_ratio": 0.15,
        "seed": 42,
        "batch_size": 4,
        "num_workers": 0
    }

trainset, valset, testset = data.dataloader.build_dataloaders("./HMDB_simp", cfg)

print(len(trainset), len(valset), len(testset))

Found 25 classes, 1250 samples
Split sizes Train: 874 | Val: 188 | Test: 188
219 47 47


## Model setup

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT = "MCG-NJU/videomae-base-finetuned-kinetics"
NUM_CLASSES = 25
model_inst = model.videomae.load_videomae(checkpoint=CHECKPOINT, num_classes=NUM_CLASSES)

Loading weights: 100%|██████████| 186/186 [00:00<00:00, 18516.14it/s]
VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400]) vs model:torch.Size([25])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400, 768]) vs model:torch.Size([25, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [4]:
model_inst = model_inst.to(device)
model_inst.eval()

# Parameter count
params = model.videomae.count_parameters(model_inst)
print(f"\nParameters:")
print(f"Total: {params['total']:,}")
print(f"Trainable : {params['trainable']:,}")

# Check the classification head has been swapped correctly
print(f"\nClassifier output features: {model_inst.classifier.out_features}")  # expect 25


Parameters:
Total: 86,246,425
Trainable : 86,246,425

Classifier output features: 25
